# A Quick Recap for the each part of the AI agents field


##  LangChain vs LangGraph State & Component Comparison

This document summarizes the key concepts discussed: **LangChain**, **LangGraph**, **AgentState**, **TypedDict State**, **dataclass Context**, and **Pydantic BaseModel**.

---

### 1. High-Level Architecture

| Layer | Purpose | Typical Components | Who Owns It |
|------|---------|-------------------|-------------|
| Capability Layer | Provides AI functionality | LLMs, Tools, Prompts | LangChain |
| Execution Layer | Controls flow of execution | Graph nodes, edges, reducers | LangGraph |
| Agent Memory | Stores agent reasoning state | AgentState | LangChain |
| Workflow State | Data passed between steps | TypedDict State | LangGraph |
| Runtime Context | External configuration | dataclass context | Application |
| Data Contracts | Validate structured input/output | Pydantic BaseModel | Tools / APIs |

---

### 2. LangChain vs LangGraph

| Feature | LangChain | LangGraph |
|-------|-----------|-----------|
| Primary Role | AI capability framework | Execution orchestration |
| Core Concept | Agents | Graph workflows |
| Control Flow | Implicit agent loop | Explicit graph |
| State Management | Agent memory | Workflow state |
| Parallel Execution | Limited | Native |
| Determinism | Lower | High |
| Production Workflows | Harder to manage | Designed for production |
| Tooling | LLMs, prompts, tools | Nodes, edges, reducers |
| Interrupt Support | Agent-level | Graph-level |
| Checkpointing | Limited | Built-in |

---

### 3. AgentState vs TypedDict State

| Aspect | AgentState | TypedDict State |
|------|-------------|----------------|
| Library | LangChain | LangGraph |
| Purpose | Agent internal memory | Workflow shared state |
| Structure | Python class | Dictionary schema |
| Mutation Style | Mutable object | Functional updates |
| Scope | Single agent | Entire workflow |
| Supports Reducers | No | Yes |
| Parallel Updates | No | Yes |
| Context Growth | Continuous | Controlled |
| Used By | Tools, middleware | Graph nodes |
| Control Flow | Agent loop | Graph edges |

---

### 4. dataclass Context vs AgentState

| Aspect | dataclass Context | AgentState |
|------|------------------|-----------|
| Purpose | Runtime configuration | Agent memory |
| Mutability | Typically static | Mutable |
| Lifecycle | Provided at invocation | Changes during execution |
| Storage | Outside the agent | Inside the agent |
| Example Data | API keys, user info | authentication flags |
| Persistence | No | Optional |
| Context Window Impact | None | Yes |


### 5. TypedDict vs BaseModel

| Aspect | TypedDict | BaseModel |
|------|------------------|-----------|
| Library | Python typing | Pydantic |
| Purpose | Describe dictionary structure | Validate structured data |
| Runtime Validation | No | Yes |
| Used In | Graph state |Tools / APIs |
| Serialization | Manual | Automatic |
| Strict Typing | Static only | Runtime enforcedOptional |
| Ideal For | Workflow data flow | API schemas  |



### 6. State Evolution Comparison

| Property        | AgentState          | TypedDict         |
| --------------- | ------------------- | ----------------- |
| Memory Pattern  | Accumulating memory | Selective updates |
| Context Size    | Grows continuously  | Controlled        |
| Data Ownership  | Agent               | Workflow          |
| Update Method   | Attribute mutation  | Return dictionary |
| Parallel Safety | No                  | Yes               |


### 7. Mental Model Summary

| Concept           | Think of it as                       |
| ----------------- | ------------------------------------ |
| LangChain         | AI capability toolkit                |
| LangGraph         | Workflow execution engine            |
| AgentState        | What the agent remembers             |
| TypedDict State   | Data flowing through the workflow    |
| dataclass Context | Configuration provided to the system |
| BaseModel         | Contract enforcing structured data   |


### 8. Final Simplified Architecture Diagram
```text
Application
   |
   |-- Context (dataclass)
   |
LangGraph Workflow
   |
   |-- State (TypedDict)
   |
   |-- Node
        |
        |-- LangChain Agent
                |
                |-- AgentState
                |-- Tools
                |-- LLM
```

### RAG From Scratch: Query Construction

Converts natural language questions into **structured database queries** with metadata filters. This enables filtering by publish date, view count, video length, etc. — going beyond pure semantic similarity search.

In [ ]:
import warnings
import os 
from dotenv import load_dotenv

# Suppress all warnings for cleaner notebook output
warnings.filterwarnings("ignore")

# Load environment variables from the .env file into the process
load_dotenv()

try: 
    # Map .env variables to the keys LangChain/LangSmith expects at runtime
    os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGSMITH_TRACING_V2")   # Enable LangSmith tracing
    os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")         # LangSmith authentication key
    os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")         # LangSmith project name for grouping traces
    os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")       # LangSmith API endpoint URL
    os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY")             # Mistral AI LLM/embedding API key
    os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")                           # HuggingFace access token
    os.environ["USER_AGENT"] = "MyLangChainApp/1.0"                           # Required User-Agent header for WebBaseLoader
    print("Environment variables set successfully")
except Exception as e: 
    print(f"Error: {e}")

Environment variables set successfully


### Environment Initialization

Loads `.env` variables for LangSmith tracing, Mistral API, HuggingFace token, and User-Agent. Suppresses warnings.

#### Part 11: Query Structuring for Metadata Filters

Many vector stores support **metadata filtering** alongside semantic search. This section demonstrates:
1. Loading a YouTube transcript with rich metadata (title, view count, publish date, length).
2. Defining a Pydantic `TutorialSearch` schema with optional filter fields.
3. Using `llm.with_structured_output()` to convert natural language queries into structured filter objects.

In [ ]:
# pytubefix is a maintained fork of pytube; register it as a shim so
# LangChain's YoutubeLoader (which imports pytube internally) works
import pytubefix
import sys
sys.modules['pytube'] = pytubefix

from langchain_community.document_loaders import YoutubeLoader

# Load transcript + video metadata (title, view_count, publish_date, length, etc.)
docs = YoutubeLoader.from_youtube_url(
    "https://youtu.be/pbAd8O1Lvm4",
    add_video_info=True,  # Fetch video metadata alongside the transcript
).load()

# Inspect the metadata dictionary — these fields will be used for structured filtering
docs[0].metadata

{'source': 'pbAd8O1Lvm4',
 'title': 'Self-reflective RAG with LangGraph: Self-RAG and CRAG',
 'description': 'Self-reflection can greatly enhance RAG, enabling correction of poor quality retrieval or generations. Several recent RAG papers focus on this theme, but implementing the ideas can be tricky. Here, we show that LangGraph can be easily used for "flow engineering" of self-reflective RAG pipelines. We provide cookbooks for implementing ideas from two interesting papers, Self-RAG and C-RAG.\n\nCode:\nhttps://github.com/langchain-ai/langgraph/tree/main/examples/rag',
 'view_count': 37509,
 'thumbnail_url': 'https://i.ytimg.com/vi/pbAd8O1Lvm4/sddefault.jpg',
 'publish_date': '2024-02-07 08:46:59',
 'length': 1058,
 'author': 'LangChain'}

#### Load YouTube Transcript & Metadata

Uses `YoutubeLoader` (via `pytubefix` compatibility shim) to load a video transcript with `add_video_info=True`. The metadata dict includes fields like `title`, `view_count`, `publish_date`, and `length` — these are the fields we'll build structured filters for.

In [ ]:
import datetime
from typing import Optional
from pydantic import BaseModel, Field

class TutorialSearch(BaseModel):
    """Search over a database of tutorial videos about a software library.
    
    The LLM populates this schema from natural language queries.
    Optional filter fields are only set when the user explicitly specifies constraints.
    """
    
    # Required: semantic search query against video transcripts
    # "..." (Ellipsis) means this field is required — no default value
    content: str = Field(
        ..., 
        description = "Similarity search query applied to video transcripts.",
    )
    # Required: shortened keyword version for matching video titles
    title_search: str = Field(
        ..., 
        description = """Alternate version of the content search query to apply to video titles. \n 
            Should be succinct and only include key words that could be in a video title."""
    )
    # Optional filters — only populated when the user explicitly specifies constraints
    min_view_count: Optional[int] = Field(
        None,                                   # Default: no filter
        description = "Minimum view count filter, inclusive. Only use if explicitly specified.",
    )
    max_view_count: Optional[int] = Field(
        None,
        description="Maximum view count filter, exclusive. Only use if explicitly specified.",
    )
    earliest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Earliest publish date filter, inclusive. Only use if explicitly specified.",
    )
    latest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Latest publish date filter, exclusive. Only use if explicitly specified.",
    )
    min_length_sec: Optional[int] = Field(
        None,
        description="Minimum video length in seconds, inclusive. Only use if explicitly specified.",
    )
    max_length_sec: Optional[int] = Field(
        None,
        description="Maximum video length in seconds, exclusive. Only use if explicitly specified."
    )
    
    def pretty_print(self) -> None: 
        """Print only non-default fields for a clean summary of active filters."""
        for field in self.__fields__:
            if getattr(self, field) is not None and getattr(self, field) != getattr(
                self.__fields__[field], "default", None
            ):
                print(f"{field}: {getattr(self, field)}")

#### Define `TutorialSearch` Pydantic Schema

A Pydantic `BaseModel` that defines the structured query format:
- **`content`** — Semantic search query for video transcripts.
- **`title_search`** — Keyword search for video titles.
- **Optional filters** — `min/max_view_count`, `earliest/latest_publish_date`, `min/max_length_sec`.

Fields use `Field(None, ...)` with descriptive prompts so the LLM only populates filters when explicitly specified by the user. The `pretty_print()` method displays only non-default fields.

#### Query Analyzer Chain

Builds the query analyzer:
- **System prompt** instructs the LLM to convert user questions into optimized database queries, preserving unfamiliar terms as-is.
- **`llm.with_structured_output(TutorialSearch)`** forces `mistral-medium-latest` to return a validated `TutorialSearch` object.
- The chain is tested with several queries of increasing complexity (basic search, date filters, view count + length filters).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai import ChatMistralAI

# System prompt: instruct the LLM to convert natural language → structured DB query
# Specifically tells it NOT to rephrase unfamiliar acronyms (e.g., "LCEL", "RAG")
system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),   # User's natural language query
    ]
)

# Initialize LLM with deterministic output and structured output enforcement
llm = ChatMistralAI(model = "mistral-medium-latest", temperature = 0)
structured_llm = llm.with_structured_output(TutorialSearch)  # Forces return of TutorialSearch object

# Query analyzer chain: prompt → structured LLM → TutorialSearch object
query_analyzer = prompt | structured_llm

# Test: simple content-only query (no filters expected)
query_analyzer.invoke({"question": "rag from scratch"}).pretty_print()

content: rag from scratch
title_search: rag from scratch


#### Test: Date-Filtered Query

Invokes the query analyzer with *"videos on chat langchain published in 2023"*. The LLM should populate `content`, `title_search`, and the date filter fields while leaving other filters as `None`.

In [ ]:
# Test: query with date filter — LLM should populate earliest/latest_publish_date
query_analyzer.invoke(
    {"question": "videos on chat langchain published in 2023"}
).pretty_print()

content: videos on chat langchain
title_search: chat langchain
earliest_publish_date: 2023-01-01
latest_publish_date: 2024-01-01


#### Test: Date Boundary Query

Tests with *"videos focused on chat langchain published before 2024"* — the LLM should set `latest_publish_date` to filter exclusively.

In [ ]:
# Test: "before 2024" — LLM should set latest_publish_date as an exclusive upper bound
query_analyzer.invoke(
    {"question": "videos that are focused on the topic of chat langchain that are published before 2024"}
).pretty_print()

content: chat langchain
title_search: chat langchain
latest_publish_date: 2024-01-01


#### Test: Length-Filtered Query

Tests with *"how to use multi-modal models in an agent, only videos under 5 minutes"* — the LLM should set `max_length_sec` to 300 alongside the content and title fields.

In [ ]:
# Test: length filter — "under 5 minutes" should set max_length_sec to 300
query_analyzer.invoke(
    {
        "question": "how to use multi-modal models in an agent, only videos under 5 minutes"
    }
).pretty_print()

content: use multi-modal models in an agent
title_search: multi-modal models agent
max_length_sec: 300


To then connect this to various vectorstores, you can follow [here](https://docs.langchain.com/oss/python/langchain/overview#constructing-from-scratch-with-lcel)